# 📋 Job & Internship Application Tracker — Automation Assignment

**Course:** Ubuntu Automation  
**Topic:** Automating Job/Internship Application Tracking  

---

## 🎯 Overview

Job hunting is exhausting — especially when you're applying to dozens of companies at once. This project automates **5 real-world tasks** a job seeker would normally do by hand:

| # | Scenario | What's Automated |
|---|----------|-------------------|
| 1 | **Status Dashboard** | Summarize all applications at a glance |
| 2 | **Follow-up Alert System** | Flag applications that need a follow-up |
| 3 | **Pipeline Funnel Visualizer** | Show where candidates drop off |
| 4 | **Salary & Platform Intelligence** | Compare offers and find best sources |
| 5 | **Weekly Activity Report Generator** | Auto-generate a markdown summary report |

All scenarios run on simulated CSV data representing a 3-month job search campaign (30 applications).

---
## ⚙️ Setup — Install Libraries & Load Data

We begin by importing all required libraries and loading the CSV dataset.
The dataset (`job_applications.csv`) contains 30 job/internship applications with fields like:
- `company`, `role`, `type` (Internship / Full-time)
- `date_applied`, `status`, `follow_up_date`, `response_date`
- `platform`, `remote`, `salary_min`, `salary_max`
- `interview_rounds`, `referral`, `cover_letter`

In [ ]:
# ── Install any missing packages (Colab usually has these) ──────────────────
# !pip install pandas matplotlib seaborn plotly --quiet

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('job_applications.csv', parse_dates=['date_applied','follow_up_date','response_date'])

# Derived columns
df['days_to_response'] = (df['response_date'] - df['date_applied']).dt.days
df['salary_mid']       = (df['salary_min'] + df['salary_max']) / 2
df['month_applied']    = df['date_applied'].dt.to_period('M').astype(str)

TODAY = pd.Timestamp('2024-04-01')   # simulated "today"

print(f'✅ Dataset loaded — {len(df)} applications across {df["company"].nunique()} companies')
df.head()

---
## 🗂️ Scenario 1 — Automated Status Dashboard

**Problem:** After applying to 30+ companies, it's easy to lose track of where each application stands.  
**Automation:** This scenario automatically categorises every application by status and renders a colour-coded summary — just like a real CRM (Customer Relationship Management) tool.

We generate:
1. A **summary table** with counts and percentages per status
2. A **donut chart** showing the overall pipeline health
3. A quick **success rate KPI** calculation

In [ ]:
# ── Colour palette mapped to emotional meaning ───────────────────────────────
STATUS_COLORS = {
    'Applied'   : '#4A90D9',
    'Interview' : '#F5A623',
    'Offer'     : '#27AE60',
    'Rejected'  : '#E74C3C',
    'Ghosted'   : '#95A5A6',
}

status_counts = df['status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']
status_counts['Percentage'] = (status_counts['Count'] / len(df) * 100).round(1)

# ── KPIs ─────────────────────────────────────────────────────────────────────
offers      = len(df[df['status'] == 'Offer'])
interviews  = len(df[df['status'].isin(['Interview','Offer'])])
total       = len(df)
offer_rate  = offers / total * 100
inter_rate  = interviews / total * 100

print('━' * 45)
print(f'  📊  APPLICATION DASHBOARD SUMMARY')
print('━' * 45)
print(f'  Total Applications  : {total}')
print(f'  Offers Received     : {offers}   ({offer_rate:.1f}%)')
print(f'  Interview Rate      : {interviews}   ({inter_rate:.1f}%)')
print('━' * 45)
print(status_counts.to_string(index=False))

# ── Donut chart ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
colors  = [STATUS_COLORS.get(s, '#BDC3C7') for s in status_counts['Status']]
wedges, texts, autotexts = ax.pie(
    status_counts['Count'],
    labels      = status_counts['Status'],
    colors      = colors,
    autopct     = '%1.1f%%',
    startangle  = 140,
    pctdistance = 0.78,
    wedgeprops  = dict(width=0.55, edgecolor='white', linewidth=2)
)
for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
ax.text(0, 0, f'{total}\nApps', ha='center', va='center', fontsize=16, fontweight='bold', color='#2C3E50')
ax.set_title('Application Pipeline — Status Overview', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('dashboard_donut.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard chart saved.')

---
## ⏰ Scenario 2 — Automated Follow-up Alert System

**Problem:** Recruiters often ghost applicants. Following up at the right time can make the difference — but remembering to do so for 30+ applications is unrealistic manually.  
**Automation:** This scenario mimics a **cron-job-style alert system** that checks each application and flags:

- 🔴 **OVERDUE** — Follow-up date has passed and status is still `Applied`
- 🟡 **DUE TODAY** — Follow-up is due on the simulated today
- 🟢 **UPCOMING** — Follow-up within the next 7 days

In a real Ubuntu environment, this script would be scheduled via `cron` to run every morning.

In [ ]:
# ── Simulate cron-job alert logic ────────────────────────────────────────────
active = df[df['status'].isin(['Applied', 'Interview'])].copy()
active['days_since_applied'] = (TODAY - active['date_applied']).dt.days
active['days_to_followup']   = (active['follow_up_date'] - TODAY).dt.days

def alert_level(row):
    d = row['days_to_followup']
    if pd.isna(d):       return '⚫ No Date Set'
    if d < 0:            return '🔴 OVERDUE'
    if d == 0:           return '🟡 DUE TODAY'
    if d <= 7:           return '🟢 UPCOMING'
    return '⚪ Future'

active['alert'] = active.apply(alert_level, axis=1)

print(f'🤖 Follow-up Alert System — Simulated run on {TODAY.date()}')
print('=' * 60)
for level in ['🔴 OVERDUE', '🟡 DUE TODAY', '🟢 UPCOMING']:
    subset = active[active['alert'] == level][['company','role','date_applied','follow_up_date','days_to_followup']]
    if not subset.empty:
        print(f'\n  {level} ({len(subset)} applications)')
        print(subset.to_string(index=False))

# ── Bar chart of alert distribution ──────────────────────────────────────────
alert_counts = active['alert'].value_counts()
alert_colors = {'🔴 OVERDUE':'#E74C3C','🟡 DUE TODAY':'#F39C12','🟢 UPCOMING':'#27AE60','⚪ Future':'#BDC3C7','⚫ No Date Set':'#7F8C8D'}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(alert_counts.index, alert_counts.values,
               color=[alert_colors.get(k,'#BDC3C7') for k in alert_counts.index],
               edgecolor='white', linewidth=1.5)
for bar in bars:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', va='center', fontweight='bold')
ax.set_xlabel('Number of Applications')
ax.set_title('Follow-up Alert Levels (Active Applications)', fontsize=13, fontweight='bold')
ax.set_xlim(0, alert_counts.max() + 2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('followup_alerts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Alert system chart saved.')

---
## 🔽 Scenario 3 — Recruitment Funnel Visualizer

**Problem:** Recruiters and job seekers both need to understand *where* candidates drop off in the process. This is standard in sales (conversion funnels) but rarely tracked by individual applicants.  
**Automation:** This scenario automatically builds a **recruitment funnel** showing how many applications progressed through each stage. It also breaks down the funnel by **Internship vs Full-time** to reveal which track has better conversion rates.

Funnel stages: `Applied → Interview → Offer`

In [ ]:
# ── Funnel progression logic ─────────────────────────────────────────────────
# Each stage is cumulative: "reached at least this stage"
STAGE_ORDER = ['Applied', 'Interview', 'Offer']

def build_funnel(data):
    counts = {'Applied': len(data)}   # all apps started at Applied
    counts['Interview'] = len(data[data['status'].isin(['Interview','Offer'])])
    counts['Offer']     = len(data[data['status'] == 'Offer'])
    return counts

overall  = build_funnel(df)
intern   = build_funnel(df[df['type'] == 'Internship'])
fulltime = build_funnel(df[df['type'] == 'Full-time'])

print('Funnel Conversion Rates:')
for stage in STAGE_ORDER:
    print(f"  {stage:12s}: Overall={overall[stage]:2d}  |  Internship={intern[stage]:2d}  |  Full-time={fulltime[stage]:2d}")

# ── Side-by-side funnel bars ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
datasets = [('Overall', overall, '#3498DB'), ('Internship', intern, '#9B59B6'), ('Full-time', fulltime, '#E67E22')]

for ax, (title, data, color) in zip(axes, datasets):
    stages = list(data.keys())
    vals   = list(data.values())
    bars   = ax.bar(stages, vals, color=color, alpha=0.85, edgecolor='white', linewidth=2, width=0.5)
    for bar, val in zip(bars, vals):
        pct = val / vals[0] * 100 if vals[0] > 0 else 0
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}\n({pct:.0f}%)', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.set_ylim(0, max(vals) + 4)
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylabel('Applications')

fig.suptitle('Recruitment Funnel — Stage Conversion Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('funnel_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Funnel chart saved.')

---
## 💰 Scenario 4 — Salary & Platform Intelligence Report

**Problem:** Job seekers rarely know which platform gets the most responses, or what salary range they're targeting on average. Without data, decisions are based on gut feeling.  
**Automation:** This scenario runs an automated analysis that:

1. Compares **average mid-salary by platform** — where are the highest-paying jobs listed?
2. Compares **response rates by platform** — which platforms actually convert to interviews/offers?
3. Visualises the **salary range distribution** as a box-plot by job type

This mirrors what a data pipeline in a company HR system would generate nightly.

In [ ]:
# ── Platform response rate ────────────────────────────────────────────────────
df['responded'] = df['status'].isin(['Interview','Offer','Rejected']).astype(int)
platform_stats = df.groupby('platform').agg(
    Applications = ('application_id','count'),
    Avg_Salary   = ('salary_mid','mean'),
    Response_Rate= ('responded','mean')
).round(2).reset_index()
platform_stats['Response_Rate_%'] = (platform_stats['Response_Rate'] * 100).round(1)
platform_stats['Avg_Salary_M']    = (platform_stats['Avg_Salary'] / 1_000_000).round(2)

print('Platform Intelligence Report')
print(platform_stats[['platform','Applications','Avg_Salary_M','Response_Rate_%']].to_string(index=False))

# ── 2-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — avg salary by platform
ps = platform_stats.sort_values('Avg_Salary_M', ascending=True)
axes[0].barh(ps['platform'], ps['Avg_Salary_M'], color='#2ECC71', edgecolor='white', linewidth=1.5)
for i, (v, p) in enumerate(zip(ps['Avg_Salary_M'], ps['platform'])):
    axes[0].text(v + 0.05, i, f'₩{v:.2f}M', va='center', fontsize=9)
axes[0].set_xlabel('Average Mid Salary (₩ Millions)')
axes[0].set_title('Avg Salary by Platform', fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Right — response rate by platform
ps2 = platform_stats.sort_values('Response_Rate_%', ascending=True)
axes[1].barh(ps2['platform'], ps2['Response_Rate_%'], color='#E74C3C', edgecolor='white', linewidth=1.5)
for i, v in enumerate(ps2['Response_Rate_%']):
    axes[1].text(v + 0.5, i, f'{v}%', va='center', fontsize=9)
axes[1].set_xlabel('Response Rate (%)')
axes[1].set_title('Response Rate by Platform', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Platform Intelligence — Where to Apply?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('platform_intelligence.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Salary box-plot by type ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='salary_mid', by='type', ax=ax,
           boxprops=dict(color='#2C3E50'),
           medianprops=dict(color='#E74C3C', linewidth=2),
           whiskerprops=dict(color='#7F8C8D'),
           flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=6))
ax.set_title('Salary Distribution: Internship vs Full-time', fontsize=13, fontweight='bold')
ax.set_xlabel('Job Type')
ax.set_ylabel('Mid Salary (₩)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('salary_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Platform intelligence charts saved.')

---
## 📝 Scenario 5 — Automated Weekly Report Generator

**Problem:** A job seeker or career coach needs a weekly summary without manually counting spreadsheet rows.  
**Automation:** This scenario auto-generates a **structured Markdown report** and a **timeline line chart** showing application activity over time — simulating what a scheduled weekly email digest would contain.

In a real Ubuntu environment, this would be called by a `cron` job every Monday morning and emailed using `sendmail` or a Python SMTP script.

In [ ]:
# ── Weekly application activity ───────────────────────────────────────────────
df['week'] = df['date_applied'].dt.to_period('W').apply(lambda r: r.start_time)
weekly = df.groupby('week').agg(
    total       = ('application_id','count'),
    offers      = ('status', lambda x: (x=='Offer').sum()),
    interviews  = ('status', lambda x: x.isin(['Interview','Offer']).sum()),
    rejections  = ('status', lambda x: (x=='Rejected').sum())
).reset_index()

# ── Timeline chart ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(weekly['week'], weekly['total'], alpha=0.15, color='#3498DB')
ax.plot(weekly['week'], weekly['total'],    'o-', color='#3498DB', linewidth=2.5, label='Applied',    markersize=7)
ax.plot(weekly['week'], weekly['interviews'],'s-', color='#F39C12', linewidth=2,   label='Interviews', markersize=7)
ax.plot(weekly['week'], weekly['offers'],   '^-', color='#27AE60', linewidth=2,   label='Offers',     markersize=8)
ax.plot(weekly['week'], weekly['rejections'],'v-', color='#E74C3C', linewidth=2,  label='Rejections', markersize=7)
ax.legend(loc='upper right', frameon=True)
ax.set_xlabel('Week')
ax.set_ylabel('Count')
ax.set_title('Weekly Application Activity Timeline', fontsize=14, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('weekly_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Auto-generate Markdown report ────────────────────────────────────────────
last_week_apps = weekly.iloc[-1]
report = f"""# 📬 Weekly Job Search Report
**Generated:** {TODAY.strftime('%B %d, %Y')}  |  **Period:** Last 3 months

---

## 📊 Overall Summary
| Metric | Value |
|--------|-------|
| Total Applications | {len(df)} |
| Internships | {len(df[df['type']=='Internship'])} |
| Full-time Roles | {len(df[df['type']=='Full-time'])} |
| Interviews Reached | {len(df[df['status'].isin(['Interview','Offer'])])} |
| Offers Received | {len(df[df['status']=='Offer'])} |
| Offer Rate | {len(df[df['status']=='Offer'])/len(df)*100:.1f}% |
| Avg Days to Response | {df['days_to_response'].mean():.1f} days |

## 🏆 Top Outcomes
"""
offers_list = df[df['status']=='Offer'][['company','role','salary_mid']]
for _, row in offers_list.iterrows():
    report += f"- ✅ **{row['company']}** — {row['role']} (₩{row['salary_mid']:,.0f}/mo)\n"

report += f"""
## ⚠️ Action Items
- {len(df[(df['status']=='Applied') & ((TODAY - df['date_applied']).dt.days > 14)])} applications need follow-up (applied 14+ days ago)
- {len(df[df['status']=='Interview'])} interviews in progress

---
*Auto-generated by Job Tracker Automation — run via cron every Monday at 08:00*
"""

# Save to file
with open('weekly_report.md', 'w') as f:
    f.write(report)

print(report)
print('✅ Weekly report saved to weekly_report.md')

---
## 💭 Reflection

> *Max 200 words — personal takeaways from this assignment.*

This assignment fundamentally changed how I think about automation. Before, I understood automation as something that happens in servers and scripts. After building this tracker, I realised automation is about **removing repeated human effort from any structured task** — including something as personal as a job search.

The most meaningful lesson came from **Scenario 2**, the follow-up alert system. What looks like a simple "check the date" operation is exactly the kind of logic that runs inside real CRM tools used by enterprise sales teams. The difference between my script and a production system is just infrastructure — the logic is the same.

I also learned that **data quality matters more than code quality**. Designing the CSV carefully — including edge cases like ghosted applications and missing response dates — forced me to write more robust `NaN`-handling code and made the visualisations more honest.

Finally, the **report generator** in Scenario 5 made me appreciate cron jobs differently. Scheduling is not just a convenience — it's what transforms a script into a *system*. I now see Ubuntu's cron as a backbone for turning one-time code into continuous automation.

This project made me a more thoughtful programmer and a more organised job seeker at the same time.